# Network Security Audit

This notebook queries the OCP audit SQLite datastore to present findings for:

- **OCP-30 — Network Port Restriction**: Access to sensitive ports is restricted to required networks and subnets. Evidence on-cluster: CNI plugin posture (egress-firewall enablement on OVN-Kubernetes), active egress firewall rules, IngressController endpoint-publishing strategy, route TLS posture, and the count of insecure (non-TLS) routes.
- **OCP-31 — Service Mesh Enforcement**: Use of a service mesh (e.g. Istio / OpenShift Service Mesh) is enforced for in-cluster traffic. Evidence on-cluster: install posture of the `servicemesh-operator`, presence of a `ServiceMeshControlPlane` (SMCP) resource, and whether SMCP mTLS is set to `strict`.
- **OCP-32 — Native Network Policies**: Kubernetes-native NetworkPolicies are used to control traffic between pods and namespaces. Evidence on-cluster: count of NetworkPolicy resources and the breakdown of namespaces that have at least one NetworkPolicy versus the total number of namespaces.


In [ ]:
import os
import re
import sys

import pandas as pd

# Ensure the repo root is importable before loading project modules.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from notebook_style import bootstrap, style_table  # noqa: E402

print("python:", sys.executable)
print("cwd:", os.getcwd())

# bootstrap() adds ../datastore to sys.path, which is required before
# importing schema.models below.
session, engine = bootstrap()

from schema.models import (  # noqa: E402
    Cluster,
    IngressBoundaryProtection,
    NetworkSecurityMesh,
)

print("Connected to:", engine.url)

## Cluster Inventory


In [ ]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

In [ ]:
def _detail_has(value, needle):
    if value is None:
        return False
    return needle in str(value)


def _extract_int(value, regex):
    if value is None:
        return None
    m = regex.search(str(value))
    return int(m.group(1)) if m else None

---
## OCP-30: Network Port Restriction

*Access to sensitive ports is restricted to required networks and subnets.*

On-cluster evidence is the **CNI plugin** posture (whether OVN-Kubernetes is active and whether the **egress firewall** feature is enabled), the count of **active egress firewall rules**, the **IngressController** posture (endpoint-publishing strategy and minimum TLS version), the breakdown of **route counts** by TLS termination, and the count of **insecure routes** (non-TLS / `insecure-edge-termination-policy=Allow`). Record types in scope: `cni`, `egress_firewall_count` from `network_security_mesh`, and `ingresscontroller`, `route_count`, `insecure_routes` from `ingress_boundary_protection`.

> **Note on scope.** The current export does not enumerate per-namespace port allowlists or NodePort services individually; the closest signals available on-cluster are surfaced here. Treat insecure-route count and absence of egress-firewall rules as the strongest indicators of an unrestricted boundary.

### Network port / boundary records (per cluster)


In [ ]:
OCP30_NSM_RECORD_TYPES = ("cni", "egress_firewall_count")
OCP30_IBP_RECORD_TYPES = ("ingresscontroller", "route_count", "insecure_routes")

df_ocp30_nsm = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type.in_(OCP30_NSM_RECORD_TYPES))
    .order_by(Cluster.cluster_name, NetworkSecurityMesh.record_type)
    .statement,
    engine,
)

df_ocp30_ibp = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        IngressBoundaryProtection.record_type,
        IngressBoundaryProtection.component_name,
        IngressBoundaryProtection.status,
        IngressBoundaryProtection.namespace,
        IngressBoundaryProtection.detail_1,
        IngressBoundaryProtection.detail_2,
        IngressBoundaryProtection.detail_3,
        IngressBoundaryProtection.detail_4,
    )
    .join(Cluster, IngressBoundaryProtection.cluster_id == Cluster.id)
    .filter(IngressBoundaryProtection.record_type.in_(OCP30_IBP_RECORD_TYPES))
    .order_by(Cluster.cluster_name, IngressBoundaryProtection.record_type)
    .statement,
    engine,
)

df_ocp30 = pd.concat([df_ocp30_nsm, df_ocp30_ibp], ignore_index=True).sort_values(
    ["cluster_name", "record_type"]
)
style_table(df_ocp30, caption="OCP-30: Raw network port / boundary records")

### OCP-30: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `egress_firewall_enabled` — `detail_2` on the `cni` row contains `egress-firewall=enabled`.
- `egress_firewall_rules` — count parsed from the `status` of the `egress_firewall_count` row (must be `> 0`).
- `ingress_available` — `ingresscontroller` row has `status = Available`.
- `ingress_tls_min_v12` — `detail_3` on the `ingresscontroller` row contains `tls-min-version=VersionTLS12` (or higher).
- `insecure_route_count` — integer value from `status` on the `insecure_routes` row (must be `0`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
_INT_RE = re.compile(r"(\d+)")

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    nsm = df_ocp30_nsm[df_ocp30_nsm["cluster_name"] == cluster_name]
    ibp = df_ocp30_ibp[df_ocp30_ibp["cluster_name"] == cluster_name]

    cni_rows = nsm[nsm["record_type"] == "cni"]
    egress_firewall_enabled = not cni_rows.empty and _detail_has(
        cni_rows.iloc[0]["detail_2"], "egress-firewall=enabled"
    )

    ef_rows = nsm[nsm["record_type"] == "egress_firewall_count"]
    if ef_rows.empty:
        egress_firewall_rules = 0
    else:
        m = _INT_RE.search(str(ef_rows.iloc[0]["status"] or ""))
        egress_firewall_rules = int(m.group(1)) if m else 0

    ic_rows = ibp[ibp["record_type"] == "ingresscontroller"]
    if ic_rows.empty:
        ingress_available = False
        ingress_tls_min_v12 = False
    else:
        ic = ic_rows.iloc[0]
        ingress_available = str(ic["status"]) == "Available"
        ingress_tls_min_v12 = _detail_has(
            ic["detail_3"], "tls-min-version=VersionTLS12"
        ) or _detail_has(ic["detail_3"], "tls-min-version=VersionTLS13")

    ir_rows = ibp[ibp["record_type"] == "insecure_routes"]
    if ir_rows.empty:
        insecure_route_count = None
    else:
        m = _INT_RE.search(str(ir_rows.iloc[0]["status"] or ""))
        insecure_route_count = int(m.group(1)) if m else None

    rows.append(
        {
            "cluster_name": cluster_name,
            "egress_firewall_enabled": egress_firewall_enabled,
            "egress_firewall_rules": egress_firewall_rules,
            "ingress_available": ingress_available,
            "ingress_tls_min_v12": ingress_tls_min_v12,
            "insecure_route_count": insecure_route_count,
        }
    )

df_ocp30_flags = pd.DataFrame(rows)
df_ocp30_flags["compliant"] = (
    df_ocp30_flags["egress_firewall_enabled"]
    & (df_ocp30_flags["egress_firewall_rules"] > 0)
    & df_ocp30_flags["ingress_available"]
    & df_ocp30_flags["ingress_tls_min_v12"]
    & df_ocp30_flags["insecure_route_count"].fillna(-1).eq(0)
)
df_ocp30_noncompliant = df_ocp30_flags[~df_ocp30_flags["compliant"]].copy()
print(
    f"{len(df_ocp30_noncompliant)} of {len(df_ocp30_flags)} cluster(s) non-compliant for OCP-30"
)
style_table(df_ocp30_noncompliant, caption="OCP-30: Non-compliant clusters")

---
## OCP-31: Service Mesh Enforcement

*Use of a service mesh (e.g. OpenShift Service Mesh / Istio) is enforced for in-cluster traffic.*

On-cluster evidence is the install posture of the **`servicemesh-operator`**, the presence of one or more **`ServiceMeshControlPlane` (SMCP)** resources, and whether the SMCP enforces **mutual TLS** in `strict` mode. Record types in scope: `operator` (filtered to `component_name = servicemesh-operator`) and `smcp` from `network_security_mesh`.

### Service mesh records (per cluster)


In [ ]:
df_ocp31 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(
        (
            (NetworkSecurityMesh.record_type == "operator")
            & (NetworkSecurityMesh.component_name == "servicemesh-operator")
        )
        | (NetworkSecurityMesh.record_type == "smcp")
    )
    .order_by(Cluster.cluster_name, NetworkSecurityMesh.record_type)
    .statement,
    engine,
)
style_table(df_ocp31, caption="OCP-31: Raw service mesh records")

### OCP-31: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `service_mesh_installed` — `operator` row for `servicemesh-operator` has `status = installed`.
- `smcp_present` — at least one `smcp` row exists for the cluster.
- `smcp_ready` — at least one `smcp` row has `status = Ready`.
- `mtls_strict` — at least one `smcp` row has `detail_2` containing `mtls=strict`.

Only clusters that fail one or more of these checks are displayed.


In [ ]:
rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp31[df_ocp31["cluster_name"] == cluster_name]

    op_rows = sub[
        (sub["record_type"] == "operator")
        & (sub["component_name"] == "servicemesh-operator")
    ]
    service_mesh_installed = (
        not op_rows.empty and str(op_rows.iloc[0]["status"]) == "installed"
    )

    smcp_rows = sub[sub["record_type"] == "smcp"]
    smcp_present = not smcp_rows.empty
    smcp_ready = bool((smcp_rows["status"] == "Ready").any()) if smcp_present else False
    mtls_strict = (
        bool(smcp_rows["detail_2"].apply(lambda v: _detail_has(v, "mtls=strict")).any())
        if smcp_present
        else False
    )

    rows.append(
        {
            "cluster_name": cluster_name,
            "service_mesh_installed": service_mesh_installed,
            "smcp_present": smcp_present,
            "smcp_ready": smcp_ready,
            "mtls_strict": mtls_strict,
        }
    )

df_ocp31_flags = pd.DataFrame(rows)
df_ocp31_flags["compliant"] = df_ocp31_flags[
    ["service_mesh_installed", "smcp_present", "smcp_ready", "mtls_strict"]
].all(axis=1)
df_ocp31_noncompliant = df_ocp31_flags[~df_ocp31_flags["compliant"]].copy()
print(
    f"{len(df_ocp31_noncompliant)} of {len(df_ocp31_flags)} cluster(s) non-compliant for OCP-31"
)
style_table(df_ocp31_noncompliant, caption="OCP-31: Non-compliant clusters")

---
## OCP-32: Native Network Policies

*Kubernetes-native NetworkPolicies are used to control traffic between pods and namespaces.*

On-cluster evidence is the count of `NetworkPolicy` resources cluster-wide and the breakdown of namespaces that have at least one NetworkPolicy versus the total number of namespaces. Record type in scope: `networkpolicy_count` from `network_security_mesh`, where `status` holds the total NetworkPolicy count, `detail_1` holds `namespaces-with-policies=N`, and `detail_2` holds `total-namespaces=M`.

### NetworkPolicy records (per cluster)


In [ ]:
df_ocp32 = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        NetworkSecurityMesh.record_type,
        NetworkSecurityMesh.component_name,
        NetworkSecurityMesh.status,
        NetworkSecurityMesh.namespace,
        NetworkSecurityMesh.detail_1,
        NetworkSecurityMesh.detail_2,
        NetworkSecurityMesh.detail_3,
        NetworkSecurityMesh.detail_4,
    )
    .join(Cluster, NetworkSecurityMesh.cluster_id == Cluster.id)
    .filter(NetworkSecurityMesh.record_type == "networkpolicy_count")
    .order_by(Cluster.cluster_name)
    .statement,
    engine,
)
style_table(df_ocp32, caption="OCP-32: Raw NetworkPolicy records")

### OCP-32: Compliance flags

Per-cluster flags derived from the raw records. A cluster is considered **compliant** when:

- `networkpolicy_total` — integer `status` from the `networkpolicy_count` row (must be `> 0`).
- `namespaces_with_policies` — integer parsed from `detail_1` (`namespaces-with-policies=N`).
- `total_namespaces` — integer parsed from `detail_2` (`total-namespaces=M`).
- `coverage_pct` — `100 * namespaces_with_policies / total_namespaces` (must be `>= 50%`).

Only clusters that fail one or more of these checks are displayed.


In [ ]:
_NS_WITH_POL_RE = re.compile(r"namespaces-with-policies=(\d+)")
_TOTAL_NS_RE = re.compile(r"total-namespaces=(\d+)")

COVERAGE_THRESHOLD_PCT = 50.0

rows = []
for cluster_name in sorted(df_clusters["cluster_name"].unique()):
    sub = df_ocp32[df_ocp32["cluster_name"] == cluster_name]
    if sub.empty:
        rows.append(
            {
                "cluster_name": cluster_name,
                "networkpolicy_total": 0,
                "namespaces_with_policies": None,
                "total_namespaces": None,
                "coverage_pct": None,
            }
        )
        continue
    r = sub.iloc[0]
    m = _INT_RE.search(str(r["status"] or ""))
    np_total = int(m.group(1)) if m else 0
    ns_with = _extract_int(r["detail_1"], _NS_WITH_POL_RE)
    ns_total = _extract_int(r["detail_2"], _TOTAL_NS_RE)
    if ns_with is not None and ns_total and ns_total > 0:
        coverage_pct = round(100.0 * ns_with / ns_total, 1)
    else:
        coverage_pct = None
    rows.append(
        {
            "cluster_name": cluster_name,
            "networkpolicy_total": np_total,
            "namespaces_with_policies": ns_with,
            "total_namespaces": ns_total,
            "coverage_pct": coverage_pct,
        }
    )

df_ocp32_flags = pd.DataFrame(rows)
df_ocp32_flags["compliant"] = (
    df_ocp32_flags["networkpolicy_total"] > 0
) & df_ocp32_flags["coverage_pct"].fillna(-1).ge(COVERAGE_THRESHOLD_PCT)
df_ocp32_noncompliant = df_ocp32_flags[~df_ocp32_flags["compliant"]].copy()
print(
    f"{len(df_ocp32_noncompliant)} of {len(df_ocp32_flags)} cluster(s) non-compliant for OCP-32 "
    f"(coverage threshold {COVERAGE_THRESHOLD_PCT:.0f}%)"
)
style_table(df_ocp32_noncompliant, caption="OCP-32: Non-compliant clusters")

In [ ]:
session.close()
print("Session closed.")